# Aula 05 - Notebook: Formas Normais (FND/FNC) e Otimização de Expressões de Segurança

Neste notebook construímos funções para extrair a Forma Normal Disjuntiva (FND / Mintermos) e a Forma Normal Conjuntiva (FNC / Cláusulas) de funções booleanas da planta de fertilizantes, comparando a complexidade computacional e o tempo de execução antes e após a simplificação.

In [ ]:
import itertools
import time
from typing import List, Callable, Dict
import pandas as pd

def extrair_formas_normais(variaveis: List[str], funcao: Callable[[Dict[str, bool]], bool]):
    mintermos = []
    maxtermos = []
    
    for combo in itertools.product([False, True], repeat=len(variaveis)):
        estado = dict(zip(variaveis, combo))
        resultado = funcao(estado)
        
        if resultado:
            # Mintermo para FND
            termo = [f"{v}" if estado[v] else f"¬{v}" for v in variaveis]
            mintermos.append("(" + " ∧ ".join(termo) + ")")
        else:
            # Maxtermo para FNC
            termo = [f"¬{v}" if estado[v] else f"{v}" for v in variaveis]
            maxtermos.append("(" + " ∨ ".join(termo) + ")")
            
    fnd_str = " ∨ ".join(mintermos) if mintermos else "FALSO"
    fnc_str = " ∧ ".join(maxtermos) if maxtermos else "VERDADEIRO"
    
    return {
        "Total_Mintermos": len(mintermos),
        "Total_Maxtermos": len(maxtermos),
        "FND": fnd_str,
        "FNC": fnc_str
    }

print("Motor de formas normais carregado.")

## Exemplo: Válvula de Gás do Secador de Fertilizantes

In [ ]:
vars_gas = ['f1', 'c1', 'p_low', 'bypass']

# Expressão Não Otimizada
def gas_nao_otimizado(st: Dict[str, bool]) -> bool:
    f1, c1, p_low, bypass = st['f1'], st['c1'], st['p_low'], st['bypass']
    t1 = f1 and c1 and (not p_low)
    t2 = f1 and c1 and p_low and bypass
    t3 = f1 and (not f1) and c1 # termo redundante
    return t1 or t2 or t3

# Expressão Otimizada por Álgebra Booleana
def gas_otimizado(st: Dict[str, bool]) -> bool:
    return st['f1'] and st['c1'] and ((not st['p_low']) or st['bypass'])

# Análise Canônica
analise = extrair_formas_normais(vars_gas, gas_nao_otimizado)
print(f"Total de Mintermos (FND): {analise['Total_Mintermos']}")
print(f"Total de Maxtermos (FNC): {analise['Total_Maxtermos']}")
print("
FND Canônica:")
print(analise['FND'])
print("
FNC Canônica:")
print(analise['FNC'])


## Benchmark de Desempenho e Validação de Equivalência Lógica

In [ ]:
# Validação de que ambas produzem o mesmo valor em todos os 2^4 estados
equivalentes = True
for combo in itertools.product([False, True], repeat=len(vars_gas)):
    st = dict(zip(vars_gas, combo))
    if gas_nao_otimizado(st) != gas_otimizado(st):
        equivalentes = False
        break

print(f"As funções são logicamente equivalentes em 100% dos estados? {equivalentes}")

# Teste de latência com 500.000 iterações de scan
N = 500000
estado_teste = {'f1': True, 'c1': True, 'p_low': False, 'bypass': False}

t0 = time.time()
for _ in range(N):
    _ = gas_nao_otimizado(estado_teste)
t_nao_otimizado = time.time() - t0

t0 = time.time()
for _ in range(N):
    _ = gas_otimizado(estado_teste)
t_otimizado = time.time() - t0

df_perf = pd.DataFrame([
    {"Implementação": "Não Otimizada (SOP Bruto)", "Tempo (s)": f"{t_nao_otimizado:.4f}", "Speedup": "1.00x"},
    {"Implementação": "Otimizada (Álgebra Booleana)", "Tempo (s)": f"{t_otimizado:.4f}", "Speedup": f"{t_nao_otimizado/t_otimizado:.2f}x"}
])
df_perf